# Linear Probing MedMNIST : BiomedCLIP vs MedVAE variants

Comparaison de la qualité des représentations en **linear probing** sur plusieurs datasets MedMNIST.

Pipeline :
1. Encodeur **gelé** → extraction de features pour tout le train/test
2. Classifieur **linéaire** entraîné sur ces features
3. Comparaison des accuracies sur 4 modèles × N datasets

Modèles comparés :
- **BiomedCLIP** — ViT-B/16 pré-entraîné sur 15M paires image-texte biomédicales (embedding 512D)
- **MedVAE original** — VAE pré-entraîné officiel (features spatiales 1×28×28 → 784D)
- **MedVAE stage 1** (perso) — fine-tuning stage 1 (features spatiales 4×28×28 → 3136D)
- **MedVAE stage 2** (perso) — fine-tuning stage 2 JEPA (features spatiales 4×28×28 → 3136D)

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import medmnist
import pandas as pd
from medmnist import INFO
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, TensorDataset

# Racine du repo (trouvée en remontant jusqu'au .git)
ROOT = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()))
sys.path.insert(0, ROOT)
sys.path.insert(0, f"{ROOT}/jepa_adaptation")
from utils.launch import bootstrap, load_config
bootstrap()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIZE = 224
device

## Datasets MedMNIST

Modifier `DATASETS` pour choisir les datasets à évaluer.

In [ ]:
DATASETS = ["bloodmnist", "pathmnist", "dermamnist", "octmnist", "tissuemnist"]
N_TRAIN = 4000
N_TEST  = 2000

def build_loaders(dataset_name):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    num_classes = len(info["label"])

    tf = transforms.Compose([
        transforms.Grayscale(1),
        transforms.ToTensor(),
        transforms.Resize((SIZE, SIZE), antialias=True),
    ])

    def _build(split):
        kw = dict(split=split, transform=tf, download=True,
                  root=os.path.expanduser("~/.medmnist"), as_rgb=False)
        try:
            return DataClass(size=SIZE, **kw)
        except TypeError:
            return DataClass(**kw)

    def _subset(ds, n):
        g = torch.Generator().manual_seed(42)
        idx = torch.randperm(len(ds), generator=g)[:min(n, len(ds))].tolist()
        return Subset(ds, idx)

    train_dl = DataLoader(
        _subset(_build("train"), N_TRAIN), batch_size=64,
        shuffle=False, num_workers=4, pin_memory=True
    )
    test_dl = DataLoader(
        _subset(_build("test"), N_TEST), batch_size=64,
        shuffle=False, num_workers=4, pin_memory=True
    )
    return train_dl, test_dl, num_classes

for ds in DATASETS:
    train_dl, test_dl, nc = build_loaders(ds)
    print(f"{ds:20s}  classes={nc}  train={len(train_dl.dataset)}  test={len(test_dl.dataset)}")

## Chargement des encodeurs

Chaque loader retourne `(encode_fn, feat_dim)` :
- `encode_fn(x)` prend un batch `[B, 1, H, W]` en [0, 1] et retourne des features `[B, D]`
- Pour **BiomedCLIP** : embedding global 512D (ViT-B/16)
- Pour **MedVAE** : carte spatiale latente aplatie (flatten) — 784D pour l'original, 3136D pour stage 1/2

In [ ]:
import open_clip

def load_biomedclip():
    model, _, _ = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False

    _norm = transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711),
    )

    def encode(x):
        x_rgb = _norm(x.repeat(1, 3, 1, 1)).to(device)
        with torch.no_grad():
            return model.encode_image(x_rgb).float()

    with torch.no_grad():
        feat_dim = encode(torch.zeros(1, 1, SIZE, SIZE)).shape[1]
    print(f"BiomedCLIP       feat_dim = {feat_dim}")
    return encode, feat_dim


def load_medvae_encoder(source):
    from models.medvae import MVAE

    if source == "original":
        ddconfig = dict(
            double_z=True, z_channels=1, resolution=512, in_channels=1, out_ch=1,
            ch=128, ch_mult=[1, 2, 4], num_res_blocks=2, attn_resolutions=[], dropout=0.0,
        )
        mvae = MVAE(ddconfig=ddconfig, embed_dim=1, spatial_dims=2,
                    apply_channel_ds=False, medvae_pretrained="medvae_4_1_2d")
    else:
        cfg = load_config(
            f"{ROOT}/jepa_adaptation/configs/stage_2.yaml",
            f"{ROOT}/jepa_adaptation/configs/model.yaml",
        )
        mc = cfg["model"]
        mvae = MVAE(
            ddconfig=mc["ddconfig"],
            embed_dim=int(mc["embed_dim"]),
            spatial_dims=2,
            apply_channel_ds=bool(mc["apply_channel_ds"]),
        )
        if source == "stage1":
            mvae.load_weights(f"{ROOT}/jepa_adaptation/outputs/stage1/best.pt")
        elif source == "stage2":
            ckpt = torch.load(
                f"{ROOT}/jepa_adaptation/outputs/stage2/best.pt", map_location="cpu"
            )["model"]
            prefix = "context_encoder.autoencoder."
            state = {k[len(prefix):]: v for k, v in ckpt.items() if k.startswith(prefix)}
            mvae.model.load_state_dict(state, strict=False)
        else:
            raise ValueError(source)

    ae = mvae.model.to(device).eval()
    for p in ae.parameters():
        p.requires_grad = False

    def encode(x):
        with torch.no_grad():
            z = ae.encode(x.to(device) * 2 - 1).mode()
        return z.flatten(1).float()

    with torch.no_grad():
        feat_dim = encode(torch.zeros(1, 1, SIZE, SIZE)).shape[1]
    print(f"MedVAE {source:8s}  feat_dim = {feat_dim}")
    return encode, feat_dim


MODELS = {
    "BiomedCLIP":       load_biomedclip,
    "MedVAE original":  lambda: load_medvae_encoder("original"),
    "MedVAE stage1":    lambda: load_medvae_encoder("stage1"),
    "MedVAE stage2":    lambda: load_medvae_encoder("stage2"),
}

## Extraction de features & Linear Probing

- `extract_features` : passe tout le loader dans l'encodeur gelé → tenseurs CPU
- `linear_probe` : entraîne une couche linéaire + AdamW sur les features, évalue sur le test

In [ ]:
LP_EPOCHS = 30
LP_LR     = 1e-2
LP_WD     = 1e-2


def extract_features(encode_fn, loader):
    """Extrait les features de tout le loader (encodeur gelé)."""
    feats, labels = [], []
    with torch.no_grad():
        for img, y in loader:
            feats.append(encode_fn(img).cpu())
            labels.append(y.view(-1).long())
    return torch.cat(feats), torch.cat(labels)


def linear_probe(train_feats, train_labels, test_feats, test_labels, num_classes):
    """Entraîne un classifieur linéaire et retourne l'accuracy sur test."""
    feat_dim = train_feats.shape[1]
    clf = nn.Linear(feat_dim, num_classes).to(device)
    optimizer = torch.optim.AdamW(clf.parameters(), lr=LP_LR, weight_decay=LP_WD)
    criterion = nn.CrossEntropyLoss()

    train_dl = DataLoader(
        TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True
    )

    clf.train()
    for _ in range(LP_EPOCHS):
        for x, y in train_dl:
            optimizer.zero_grad()
            criterion(clf(x.to(device)), y.to(device)).backward()
            optimizer.step()

    clf.eval()
    with torch.no_grad():
        preds = clf(test_feats.to(device)).argmax(1).cpu()
    return (preds == test_labels).float().mean().item()

## Évaluation

Pour chaque modèle (chargé une seule fois), on extrait les features de tous les datasets et on entraîne un classifieur linéaire.

In [ ]:
all_results = {}

for model_name, load_fn in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Modèle : {model_name}")
    encode_fn, feat_dim = load_fn()
    all_results[model_name] = {}

    for ds_name in DATASETS:
        train_dl, test_dl, num_classes = build_loaders(ds_name)

        print(f"  [{ds_name}]  extraction...", end=" ", flush=True)
        train_feats, train_labels = extract_features(encode_fn, train_dl)
        test_feats,  test_labels  = extract_features(encode_fn, test_dl)

        print(f"linear probe ({LP_EPOCHS} epochs)...", end=" ", flush=True)
        acc = linear_probe(train_feats, train_labels, test_feats, test_labels, num_classes)
        all_results[model_name][ds_name] = acc
        print(f"acc = {acc:.4f}")

print("\nDone.")

## Résultats

In [ ]:
df = pd.DataFrame(all_results).T
df.index.name = "model"
df = df[DATASETS]
df["mean"] = df.mean(axis=1)

df.round(4).style.highlight_max(axis=0, color="lightgreen")

In [ ]:
n_models  = len(all_results)
n_datasets = len(DATASETS)
w = 0.8 / n_models
x = np.arange(n_datasets)
colors = plt.cm.tab10(np.linspace(0, 0.5, n_models))

fig, ax = plt.subplots(figsize=(n_datasets * 2 + 3, 5))
for i, (model_name, ds_accs) in enumerate(all_results.items()):
    accs = [ds_accs[ds] for ds in DATASETS]
    ax.bar(x + (i - n_models / 2 + 0.5) * w, accs, width=w,
           label=model_name, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(DATASETS, rotation=20, ha="right")
ax.set_ylabel("Accuracy (linear probing)")
ax.set_title("Linear Probing : BiomedCLIP vs MedVAE variants")
ax.legend(loc="upper right")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()